In [ ]:
import numpy as np
from epic.core.compilation.qec_compiler import QECCompiler
from epic.core.data_structure.pauli import PauliEigenState
from epic.core.language.qec_gadget import AllocCode, FreeCode
from epic.modules.qec_gadgets.logical_resets.init_code import InitCode
from epic.modules.qec_gadgets.readout_code import ReadoutCode
from epic.modules.qec_gadgets.transversal_gates.transversal_h import TransversalH
from epic.modules.stabilizers_codes.css_code import CSSCode

# Steane [[7,1,3]] code  –  Hx = Hz = parity-check matrix of the [7,4,3] Hamming code
H = np.array(
    [[0, 0, 0, 1, 1, 1, 1],
     [0, 1, 1, 0, 0, 1, 1],
     [1, 0, 1, 0, 1, 0, 1]],
    dtype=np.uint8,
)
lX = lZ = [1, 1, 1, 0, 0, 0, 0]

# Two independent copies – different code_name so their Tanner-graph node tags
# remain disjoint when the cone code merges both systems.
steane1 = CSSCode.from_css_pcm(code_name="steaneCode", hx=H, hz=H, logical_qubits=[(lX, lZ)])
config = {
    "objective_distance": steane1.d,
    "primitives": {
        "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
        "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
        "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
    },
}


program = [
    AllocCode(target_code=steane1, code_varname="steane1", logical_qubits_varnames=["lq_0"]),
    # Initialise both codes in |0_L> (X+ eigenstate)
    InitCode(targets=["steane1"], initial_state=PauliEigenState.X_plus),
    TransversalH(targets=["steane1"]),
    # Destructive single-code readouts to verify individual logical states
    ReadoutCode(targets=["steane1"], tag="output"),
    FreeCode(code_varname="steane1"),
]

compiler = QECCompiler(config=config)
compiled_program = compiler.compile(program)

for i in compiled_program.to_stim_program([["readout_output"]]).splitlines():
    print(i)

RX 0 2 1 6 4 5 3
RZ 10 9 8 7 11 12
REPEAT 3 {
    H 9 11 12
    TICK
    CX 12 1 9 4 11 5
    TICK
    CX 12 0 11 4 9 3
    TICK
    CX 9 0 11 2 12 4
    TICK
    CX 12 2 11 6 9 5
    TICK
    H 9 11 12
    TICK
    CX 2 8 6 10 3 7
    TICK
    CX 1 8 2 10 0 7
    TICK
    CX 4 10 0 8 5 7
    TICK
    CX 4 8 5 10
    TICK
    CX 4 7
    TICK
    MRZ 10 9 8 7 11 12
}
H 0 2 1 6 4 5 3
RZ 10 9 8 7 11 12
REPEAT 3 {
    H 9 11 12
    TICK
    CX 12 1 9 4 11 5
    TICK
    CX 12 0 11 4 9 3
    TICK
    CX 9 0 11 2 12 4
    TICK
    CX 12 2 11 6 9 5
    TICK
    H 9 11 12
    TICK
    CX 2 8 6 10 3 7
    TICK
    CX 1 8 2 10 0 7
    TICK
    CX 4 10 0 8 5 7
    TICK
    CX 4 8 5 10
    TICK
    CX 4 7
    TICK
    MRZ 10 9 8 7 11 12
}
MZ 0 6 1 5 2 3 4
DETECTOR rec[-43] rec[-37] # _det_c_z_0_steaneCode_r0_1
DETECTOR rec[-37] rec[-31] # _det_c_z_0_steaneCode_r1_2
DETECTOR rec[-42] # all_neigbors_reset_c_x_1_steaneCode
DETECTOR rec[-42] rec[-36] # _det_c_x_1_steaneCode_r0_1
DETECTOR rec[-36] rec[

In [5]:
from epic.core.experiment.noise_model import StimLikeNoiseModel
import stim

def make_noise_model(p: float) -> StimLikeNoiseModel:
    return StimLikeNoiseModel.from_stim_like_probabilities(
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )



def circ(p: float) -> stim.Circuit:
    return stim.Circuit(
        compiled_program.to_stim_program([["readout_output"]], noise_model=make_noise_model(p))
    )

In [6]:
p = circ(0.001)
for i in p:
    print(i)

len(circ(0.001).shortest_graphlike_error(ignore_ungraphlike_errors=True))

RX 0 2 1 6 4 5 3
X_ERROR(0.001) 0 2 1 6 4 5 3
R 10 9 8 7 11 12
X_ERROR(0.001) 10 9 8 7 11 12
stim.CircuitRepeatBlock(3, stim.Circuit('''
    H 9 11 12
    DEPOLARIZE1(0.001) 9 11 12
    TICK
    CX 12 1 9 4 11 5
    DEPOLARIZE2(0.001) 12 1 9 4 11 5
    TICK
    CX 12 0 11 4 9 3
    DEPOLARIZE2(0.001) 12 0 11 4 9 3
    TICK
    CX 9 0 11 2 12 4
    DEPOLARIZE2(0.001) 9 0 11 2 12 4
    TICK
    CX 12 2 11 6 9 5
    DEPOLARIZE2(0.001) 12 2 11 6 9 5
    TICK
    H 9 11 12
    DEPOLARIZE1(0.001) 9 11 12
    TICK
    CX 2 8 6 10 3 7
    DEPOLARIZE2(0.001) 2 8 6 10 3 7
    TICK
    CX 1 8 2 10 0 7
    DEPOLARIZE2(0.001) 1 8 2 10 0 7
    TICK
    CX 4 10 0 8 5 7
    DEPOLARIZE2(0.001) 4 10 0 8 5 7
    TICK
    CX 4 8 5 10
    DEPOLARIZE2(0.001) 4 8 5 10
    TICK
    CX 4 7
    DEPOLARIZE2(0.001) 4 7
    TICK
    X_ERROR(0.001) 10 9 8 7 11 12
    MR 10 9 8 7 11 12
    X_ERROR(0.001) 10 9 8 7 11 12
'''))
H 0 2 1 6 4 5 3
DEPOLARIZE1(0.001) 0 2 1 6 4 5 3
R 10 9 8 7 11 12
X_ERROR(0.001) 10 9 8 7 11

2